load the dataset, handle missing values, and examine its basic structure and properties


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import *
warnings.filterwarnings('ignore')

plt.style.use('fivethirtyeight')
sns.set(rc={'figure.figsize':(12, 8)})

# Load the data
df = pd.read_parquet('data.parquet')

df.head()


print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Number of days: {df.index.date}")

print("\nMissing values before handling:")
print(df.isnull().sum())

df = df.interpolate()
print("\nMissing values after fill:")
print(df.isnull().sum())


print("\nSummary statistics:")
df.describe()

In [ ]:
df['spread'] = df['banknifty'] - df['nifty']

#resample to daily timeframe
daily_df = df.resample('D').last().dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

# plotting Nifty IV
axes[0].plot(daily_df.index, daily_df['nifty'], color='blue', linewidth=1.5)
axes[0].set_title('Nifty IV', fontsize=14)
axes[0].set_ylabel('IV Value')
axes[0].grid(True, alpha=0.3)

# plotting Bank Nifty IV
axes[1].plot(daily_df.index, daily_df['banknifty'], color='green', linewidth=1.5)
axes[1].set_title('Bank Nifty IV', fontsize=14)
axes[1].set_ylabel('IV Value')
axes[1].grid(True, alpha=0.3)

# plotting Spread
axes[2].plot(daily_df.index, daily_df['spread'], color='red', linewidth=1.5)
axes[2].set_title('Spread', fontsize=14)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Spread Value')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot rolling statistics
window_size = 20
plt.figure(figsize=(14, 8))
plt.plot(daily_df.index, daily_df['spread'], label='Spread', color='blue', alpha=0.6)
plt.plot(daily_df.index, daily_df['spread'].rolling(window=window_size).mean(), 
         label=f'{window_size}-day Rolling Mean', color='red')
plt.plot(daily_df.index, daily_df['spread'].rolling(window=window_size).std(), 
         label=f'{window_size}-day Rolling Std', color='green')
plt.title('Rolling Statistics of IV Spread', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#identifying outliers using z-score
z_scores = (df['spread'] - df['spread'].mean()) / df['spread'].std()
outliers = df[abs(z_scores) > 3]
print(f"Number of outliers (|z| > 3): {len(outliers)}")
display(outliers.head(10) if len(outliers) > 0 else "No outliers found")

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

#ADF test to check stationarity
def adf_test(series, title=''):
    result = adfuller(series.dropna())
    print(f'ADF Test on {title}:')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.4f}')
    # If p-value < 0.05, we reject the null hypothesis (series is stationary)
    print(f'Conclusion: Series is {"stationary" if result[1] < 0.05 else "non-stationary"}')
    return result[1] < 0.05

#  KPSS test to check stationarity
def kpss_test(series, title=''):
    result = kpss(series.dropna())
    print(f'KPSS Test on {title}:')
    print(f'KPSS Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[3].items():
        print(f'\t{key}: {value:.4f}')
    # If p-value < 0.05, we reject the null hypothesis (series is non-stationary)
    print(f'Conclusion: Series is {"non-stationary" if result[1] < 0.05 else "stationary"}')
    return result[1] >= 0.05

# Perform stationarity tests on original series
print("Stationarity Tests on Original Series:\n")
nifty_stationary = adf_test(df['nifty'], 'Nifty IV')
print("\n")
banknifty_stationary = adf_test(df['banknifty'], 'Bank Nifty IV')
print("\n")
spread_stationary = adf_test(df['spread'], 'Spread')
print("\n")

# KPSS Test on spread
spread_stationary_kpss = kpss_test(df['spread'], 'Spread')
print("\n")

# If series are non-stationary, try first differencing
if not spread_stationary:
    print("Testing first difference of spread for stationarity:\n")
    df['spread_diff'] = df['spread'].diff().dropna()
    spread_diff_stationary = adf_test(df['spread_diff'], 'First Difference of Spread')
    print("\n")
    
# Plot ACF and PACF for the spread
plt.figure(figsize=(14, 6))
plt.subplot(121)
plot_acf(df['spread'].dropna(), ax=plt.gca(), lags=50)
plt.title('Autocorrelation Function (ACF) of Spread')

plt.subplot(122)
plot_pacf(df['spread'].dropna(), ax=plt.gca(), lags=50)
plt.title('Partial Autocorrelation Function (PACF) of Spread')

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.stattools import coint

data = df[['nifty', 'banknifty']].interpolate()

# Engle-Granger two-step cointegration test
eg_result = coint(data['nifty'], data['banknifty'])
print("Engle-Granger Cointegration Test:")
print(f"t-statistic: {eg_result[0]:.4f}")
print(f"p-value: {eg_result[1]:.4f}")
print(f"Critical values: {eg_result[2]}")
print(f"Conclusion: Series are {'cointegrated' if eg_result[1] < 0.05 else 'not cointegrated'} according to Engle-Granger test")

# Johansen cointegration test
try:
    daily_data = data.resample('D').last().dropna()

    johansen_result = coint_johansen(daily_data, det_order=0, k_ar_diff=1)
    
    trace_stat = johansen_result.lr1
    trace_crit = johansen_result.cvt
    
    max_eig_stat = johansen_result.lr2
    max_eig_crit = johansen_result.cvm
    
    print("\nJohansen Cointegration Test (Trace Statistic):")
    for i, (stat, crit) in enumerate(zip(trace_stat, trace_crit[:, 1])):
        print(f"H0: r<={i} vs H1: r>{i} - Test statistic: {stat:.4f}, Critical value (5%): {crit:.4f}")
        print(f"Conclusion: {'Reject' if stat > crit else 'Do not reject'} H0")
    
    print("\nJohansen Cointegration Test (Maximum Eigenvalue Statistic):")
    for i, (stat, crit) in enumerate(zip(max_eig_stat, max_eig_crit[:, 1])):
        print(f"H0: r={i} vs H1: r={i+1} - Test statistic: {stat:.4f}, Critical value (5%): {crit:.4f}")
        print(f"Conclusion: {'Reject' if stat > crit else 'Do not reject'} H0")
    
  
    if trace_stat[0] > trace_crit[0, 1]:
        eigenvectors = johansen_result.evec
        print("\nCointegration vector:")
        print(eigenvectors[:, 0])
        
        
        daily_data['coint_series'] = daily_data['nifty'] * eigenvectors[0, 0] + daily_data['banknifty'] * eigenvectors[1, 0]
        
        
        print("\nStationarity test of the cointegrated series:")
        adf_test(daily_data['coint_series'], 'Cointegrated Series')
        
        
        plt.figure(figsize=(14, 8))
        plt.plot(daily_data.index, daily_data['coint_series'])
        plt.title('Cointegrated Series', fontsize=16)
        plt.xlabel('Date')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except:
    print("Error in Johansen cointegration test. This might be due to matrix singularity or other numerical issues.")

In [ ]:

df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter

plt.figure(figsize=(14, 8))
sns.boxplot(x='hour', y='spread', data=df)
plt.title('Intraday Spread Patterns', fontsize=16)
plt.xlabel('Hour of Day')
plt.ylabel('Spread Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()




hour_groups = [df[df['hour'] == hour]['spread'].dropna() for hour in range(9, 16)]  # Trading hours
f_stat, p_val = f_oneway(*hour_groups)
print(f"ANOVA for intraday effect - F-statistic: {f_stat:.4f}, p-value: {p_val:.4f}")
print(f"Conclusion: {'Significant' if p_val < 0.05 else 'No significant'} intraday effect")



In [ ]:
spread_stats = df['spread'].describe()
print("Spread Statistics:")
print(spread_stats)


print(f"Skewness: {df['spread'].skew():.4f}")
print(f"Kurtosis: {df['spread'].kurt():.4f}")

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['spread'], kde=True)
plt.title('Histogram of Spread', fontsize=14)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
import scipy.stats as stats
stats.probplot(df['spread'].dropna(), plot=plt)
plt.title('Q-Q Plot of Spread', fontsize=14)

plt.tight_layout()
plt.show()



shapiro_test = shapiro(df['spread'].dropna())
normal_test = normaltest(df['spread'].dropna())

print(f"Shapiro-Wilk Test - Statistic: {shapiro_test[0]:.4f}, p-value: {shapiro_test[1]:.4f}")
print(f"D'Agostino's K^2 Test - Statistic: {normal_test[0]:.4f}, p-value: {normal_test[1]:.4f}")
print(f"Conclusion: Spread is {'not ' if shapiro_test[1] < 0.05 else ''}normally distributed according to Shapiro-Wilk test")

# Analyze spread behavior during different market regimes
df['nifty_percentile'] = pd.qcut(df['nifty'], 4, labels=['Low Vol', 'Medium-Low Vol', 'Medium-High Vol', 'High Vol'])

plt.figure(figsize=(14, 8))
sns.boxplot(x='nifty_percentile', y='spread', data=df)
plt.title('Spread Distribution Across Different Volatility Regimes', fontsize=16)
plt.xlabel('Market Volatility Regime')
plt.ylabel('Spread Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics of spread by market regime
print("Spread Statistics by Market Regime:")
display(df.groupby('nifty_percentile')['spread'].describe())

In [ ]:
def analyze_volatility(df, timeframe='both'):
    
    daily_df = df.resample('D').last().dropna()
    
    if timeframe in ['daily', 'both']:
        daily_windows = [5, 21, 63]  
        for window in daily_windows:
            daily_df[f'spread_std_{window}d'] = daily_df['spread'].rolling(window).std()
        
        daily_lambdas = [0.94, 0.96, 0.98]
        for lambda_param in daily_lambdas:
            daily_df[f'spread_ewma_{int(lambda_param*100)}'] = (
                daily_df['spread'].ewm(alpha=1-lambda_param).std()
            )
        
        plt.figure(figsize=(14, 8))
        for window in daily_windows:
            plt.plot(daily_df.index, 
                    daily_df[f'spread_std_{window}d'], 
                    label=f'{window}-day Rolling Std')
        plt.title('Daily Rolling Volatility of Spread', fontsize=16)
        plt.xlabel('Date')
        plt.ylabel('Standard Deviation')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        plt.figure(figsize=(14, 8))
        for lambda_param in daily_lambdas:
            plt.plot(daily_df.index, 
                    daily_df[f'spread_ewma_{int(lambda_param*100)}'],
                    label=f'Daily EWMA (λ={lambda_param})')
        plt.title('Daily EWMA Volatility of Spread', fontsize=16)
        plt.xlabel('Date')
        plt.ylabel('EWMA Standard Deviation')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    if timeframe in ['intraday', 'both']:
        df['hour'] = df.index.hour
        df['minute'] = df.index.minute
        
        trading_mask = (df['hour'].between(9, 15)) & \
                      ~((df['hour'] == 9) & (df['minute'] < 15)) & \
                      ~((df['hour'] == 15) & (df['minute'] > 30))
        trading_data = df[trading_mask]
        
        intraday_windows = [15, 30, 60]  # 15min, 30min, 60min windows
        for window in intraday_windows:
            trading_data[f'spread_std_{window}min'] = trading_data['spread'].rolling(window).std()
        
        intraday_lambdas = [0.90, 0.94, 0.96]
        for lambda_param in intraday_lambdas:
            trading_data[f'spread_ewma_intraday_{int(lambda_param*100)}'] = (
                trading_data['spread'].ewm(alpha=1-lambda_param).std()
            )
        
        time_grouped = trading_data.groupby(['hour', 'minute']).agg({
            'spread_std_15min': ['mean', 'std'],
            'spread_std_30min': ['mean', 'std'],
            'spread_std_60min': ['mean', 'std']
        }).reset_index()
        
        time_grouped['time'] = pd.to_datetime(
            time_grouped['hour'].astype(str) + ':' + 
            time_grouped['minute'].astype(str),
            format='%H:%M'
        ).dt.time
        
        fig, axes = plt.subplots(2, 2, figsize=(20, 16))
        
        for window in intraday_windows:
            axes[0,0].plot(range(len(time_grouped)), 
                         time_grouped[f'spread_std_{window}min']['mean'],
                         label=f'{window}-min Rolling Std')
        axes[0,0].set_title('Average Intraday Volatility Pattern', fontsize=14)
        axes[0,0].set_xticks(range(0, len(time_grouped), 30))
        axes[0,0].set_xticklabels([str(t) for t in time_grouped['time'][::30]], rotation=45)
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)
        
        for window in intraday_windows:
            axes[0,1].plot(range(len(time_grouped)), 
                         time_grouped[f'spread_std_{window}min']['std'],
                         label=f'{window}-min Std of Vol')
        axes[0,1].set_title('Intraday Volatility of Volatility', fontsize=14)
        axes[0,1].set_xticks(range(0, len(time_grouped), 30))
        axes[0,1].set_xticklabels([str(t) for t in time_grouped['time'][::30]], rotation=45)
        axes[0,1].legend()
        axes[0,1].grid(True, alpha=0.3)
        
        sns.boxplot(x='hour', y='spread_std_30min', data=trading_data, ax=axes[1,0])
        axes[1,0].set_title('Volatility Distribution by Hour', fontsize=14)
        axes[1,0].grid(True, alpha=0.3)
        
        first_hour = trading_data[trading_data['hour'] == 9]['spread_std_30min']
        last_hour = trading_data[trading_data['hour'] == 15]['spread_std_30min']
        
        sns.kdeplot(data=first_hour, ax=axes[1,1], label='First Hour (9:15-10:00)')
        sns.kdeplot(data=last_hour, ax=axes[1,1], label='Last Hour (14:00-15:30)')
        axes[1,1].set_title('Opening vs Closing Hour Volatility', fontsize=14)
        axes[1,1].grid(True, alpha=0.3)
        axes[1,1].legend()
        
        plt.tight_layout()
        plt.show()
        

        print("\nIntraday Volatility Statistics:")
        print("\n1. Hour-wise Average Volatility:")
        print(trading_data.groupby('hour')['spread_std_30min'].mean())
        
        print("\n2. Most/Least Volatile Periods:")
        volatility_by_time = trading_data.groupby(['hour', 'minute'])['spread_std_30min'].mean()
        print("\nLeast Volatile 5 Minutes:")
        print(volatility_by_time.nsmallest(5))
        print("\nMost Volatile 5 Minutes:")
        print(volatility_by_time.nlargest(5))

 
    daily_vol = daily_df['spread_std_21d']
    high_vol_threshold = daily_vol.quantile(0.66)
    low_vol_threshold = daily_vol.quantile(0.33)

    df['vol_regime'] = 'Medium'
    df.loc[df.index.isin(daily_df[daily_vol >= high_vol_threshold].index), 'vol_regime'] = 'High'
    df.loc[df.index.isin(daily_df[daily_vol <= low_vol_threshold].index), 'vol_regime'] = 'Low'

    print("\nVolatility Regime Statistics:")
    print("\nNumber of periods in each volatility regime:")
    print(df['vol_regime'].value_counts())
    print("\nAverage spread by volatility regime:")
    print(df.groupby('vol_regime')['spread'].mean())

analyze_volatility(df, timeframe='both')